# Terra — Family Transfer Learning (D.4)

5-class (4 family + 1 unknown) transfer on a **frozen** MobileNetV3-Large backbone.  **Colab GPU 런타임** 에서 실행.

**입력**: `family_v1.npz` (D.3 산출물, 이미 CWT+JET-LUT 렌더된 `(N,224,224,3) uint8`) + 백본 `.pth`.

**백본**: D.4 플랜 = **v3 best.pth** (170-class — top-1 낮은 건 다클래스 트레이드오프, 전이 feature 는 더 일반적).  fallback = `mobilenet_v3_large_v2_best.pth`.  `BACKBONE` 경로만 바꿔 둘 다 비교 학습 가능.

**산출물**: `mobilenet_v3_family_v1.onnx` (opset 13 단일 파일, `input`/`logits`) → Jetson `trtexec --fp16`.

> ⚠️ 정규화는 추론(`jetson_infer.to_input`)과 **bit 동일**: `/255 → (x-ImageNet_mean)/ImageNet_std`, `(1,3,224,224)`.

> ▶ 실행 전: **런타임 → 런타임 유형 변경 → GPU** 선택.

## 0. Drive 마운트 + 경로 설정
`family_v1.npz` 와 백본 `.pth` 를 Drive 에 올려두고 아래 경로만 맞추면 됨.  (Drive 안 쓰면 이 셀 대신 다음 `files.upload()` 셀 사용 후 경로를 파일명만으로 지정.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# !ls '/content/drive/MyDrive/terra'   # 경로 확인용 — 주석 풀어서 파일명 확인

In [ ]:
# ===== Config — 경로 2개만 본인 Drive 에 맞게 수정 =====
DATA_NPZ = '/content/drive/MyDrive/terra/family_v1.npz'
BACKBONE = '/content/drive/MyDrive/terra/mobilenet_v3_large_v3_best.pth'  # v3 .pth 실제 경로. 없으면 v2_best.pth
OUT_ONNX = 'mobilenet_v3_family_v1.onnx'

NUM_CLASSES   = 5         # 0~3 family, 4 unknown
EPOCHS        = 10
BATCH         = 64
LR            = 1e-3
LABEL_SMOOTH  = 0.05
OPSET         = 13
SEED          = 0

import os, numpy as np, torch
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert os.path.exists(DATA_NPZ), 'DATA_NPZ 경로 확인: ' + DATA_NPZ
assert os.path.exists(BACKBONE), 'BACKBONE 경로 확인: ' + BACKBONE
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE != 'cuda':
    print('  [!] GPU 아님 — 런타임 유형을 GPU 로 바꾸세요 (학습은 CPU 도 되지만 느림)')
print('data:', DATA_NPZ)
print('backbone:', BACKBONE)

### (대안) Drive 안 쓰고 직접 업로드
위 Drive 셀 대신 이 셀로 올린 뒤, config 의 `DATA_NPZ`/`BACKBONE` 를 **파일명만** (`'family_v1.npz'`) 으로 바꾸면 됨.

In [ ]:
# from google.colab import files
# files.upload()   # family_v1.npz + 백본 .pth 선택

## 1. 데이터 로드 + 클래스 분포

In [ ]:
z = np.load(DATA_NPZ, allow_pickle=True)
X_train, y_train = z['X_train'], z['y_train'].astype(np.int64)
X_val,   y_val   = z['X_val'],   z['y_val'].astype(np.int64)
HAS_TEST = 'X_test' in z.files
if HAS_TEST:
    X_test, y_test = z['X_test'], z['y_test'].astype(np.int64)
else:
    X_test = y_test = None
    print('[warn] 이 npz 에는 X_test 가 없음 — 구버전(per-footstep split). test 평가는 건너뜀.')
pid_map = dict(z['pid_map'])               # {label: pid_name}
CLASS_NAMES = [str(pid_map.get(i, 'cls%d' % i)) for i in range(NUM_CLASSES)]
if (NUM_CLASSES - 1) not in pid_map:
    CLASS_NAMES[NUM_CLASSES - 1] = 'unknown'

print('X_train', X_train.shape, X_train.dtype, '| X_val', X_val.shape,
      ('| X_test ' + str(X_test.shape) if HAS_TEST else '| (no test)'))
print('classes:', {i: CLASS_NAMES[i] for i in range(NUM_CLASSES)})
splits = [('train', y_train), ('val', y_val)] + ([('test', y_test)] if HAS_TEST else [])
for split, y in splits:
    u, c = np.unique(y, return_counts=True)
    print(split, {CLASS_NAMES[int(k)]: int(v) for k, v in zip(u, c)})

## 2. Dataset / DataLoader — 추론과 동일한 정규화

In [ ]:
from torch.utils.data import Dataset, DataLoader

# jetson_infer.to_input 과 동일 (ImageNet mean/std, /255, CHW)
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def preprocess(img_uint8_hwc):
    x = torch.from_numpy(np.ascontiguousarray(img_uint8_hwc)).float().permute(2, 0, 1) / 255.0
    return (x - _MEAN) / _STD

class FootstepDS(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return preprocess(self.X[i]), int(self.y[i])

train_loader = DataLoader(FootstepDS(X_train, y_train), batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(FootstepDS(X_val,   y_val),   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(FootstepDS(X_test,  y_test),  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True) if HAS_TEST else None
print('train batches', len(train_loader), '| val batches', len(val_loader),
      ('| test batches ' + str(len(test_loader)) if HAS_TEST else '| (no test)'))

## 3. 백본 로드 + freeze + head 교체 (1280 → 5)

In [ ]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_large

ckpt = torch.load(BACKBONE, map_location='cpu', weights_only=False)
# 체크포인트 포맷 robust 처리: dict 래핑(state_dict/model) 또는 raw state_dict
if isinstance(ckpt, dict) and 'state_dict' in ckpt:
    sd = ckpt['state_dict']; print('  ckpt meta:', {k: ckpt.get(k) for k in ('arch','num_classes','val_acc') if k in ckpt})
elif isinstance(ckpt, dict) and 'model' in ckpt:
    sd = ckpt['model']
else:
    sd = ckpt
orig_nc = int(sd['classifier.3.weight'].shape[0])
print('  backbone original num_classes =', orig_nc, '(v2=100, v3=170)')

model = mobilenet_v3_large(weights=None)
in_f = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_f, orig_nc)
model.load_state_dict(sd, strict=True)
print('  loaded backbone weights (strict=True)')

# 전체 freeze 후 head 만 새로
for p in model.parameters():
    p.requires_grad = False
model.classifier[3] = nn.Linear(in_f, NUM_CLASSES)   # 새 head, requires_grad=True
model = model.to(DEVICE)

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print('  trainable params:', trainable)

## 4. 학습 — class-weight 로 unknown 불균형 보정
unknown(VIBeID+noise) 이 family 의 ~2.5배라 inverse-frequency class weight 로 보정 (전량 사용).

In [ ]:
u, c = np.unique(y_train, return_counts=True)
w = np.zeros(NUM_CLASSES, dtype=np.float32)
w[u] = (len(y_train) / (len(u) * c)).astype(np.float32)   # inverse freq, 평균 1 근처
class_w = torch.tensor(w, device=DEVICE)
print('class weights:', {CLASS_NAMES[i]: round(float(w[i]), 3) for i in range(NUM_CLASSES)})

criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = total = 0
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        pred = model(xb).argmax(1).cpu().numpy()
        yb = yb.numpy()
        for t, p in zip(yb, pred):
            cm[t, p] += 1
        correct += int((pred == yb).sum()); total += len(yb)
    return correct / max(total, 1), cm

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        run_loss += loss.item() * len(yb)
    val_acc, _ = evaluate(val_loader)
    print('epoch %2d/%d  loss=%.4f  val_acc=%.4f' % (epoch, EPOCHS, run_loss / len(y_train), val_acc))

## 5. 검증 — val accuracy + per-class confusion matrix

In [ ]:
import matplotlib.pyplot as plt
val_acc, cm = evaluate(val_loader)
print('final val accuracy: %.4f' % val_acc)
print('\nper-class recall:')
for i in range(NUM_CLASSES):
    tot = cm[i].sum()
    print('  %-10s %.3f  (%d/%d)' % (CLASS_NAMES[i], cm[i, i] / max(tot, 1), cm[i, i], tot))

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title('val confusion (acc %.3f)' % val_acc)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() * 0.5 else 'black')
fig.colorbar(im); plt.tight_layout(); plt.show()

## 5b. 최종 TEST 평가 (held-out · 1회만)

페이스별 녹음의 시간 뒷부분(15%) — 학습·모델선택에 안 쓴 데이터. **val 과 test 간격이 작아야 일반화 신뢰** (크면 과적합/누수 의심).

In [ ]:
# [주의] TEST 는 학습/모델선택에 쓰지 않은 held-out (페이스별 시간 뒷부분). 최종 보고용 1회 평가.
if HAS_TEST:
    test_acc, cm_t = evaluate(test_loader)
    print('FINAL TEST accuracy: %.4f   (val_acc=%.4f 와 비교)' % (test_acc, val_acc))
    print('\nper-class recall (test):')
    for i in range(NUM_CLASSES):
        tot = cm_t[i].sum()
        print('  %-10s %.3f  (%d/%d)' % (CLASS_NAMES[i], cm_t[i, i] / max(tot, 1), cm_t[i, i], tot))
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.imshow(cm_t, cmap='Greens')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm_t[i, j], ha='center', va='center',
                    color='white' if cm_t[i, j] > cm_t.max() * 0.5 else 'black', fontsize=9)
    ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title('TEST confusion (acc %.3f)' % test_acc)
    plt.tight_layout(); plt.show()
else:
    print('X_test 없음 — prep_transfer_dataset.py 를 새로 돌려 test 포함 npz 를 만들 것.')

## 6. ONNX export (opset 13, 단일 파일) — TRT 빌드용

In [ ]:
model.eval().to('cpu')
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy, OUT_ONNX,
    input_names=['input'], output_names=['logits'],
    opset_version=OPSET,
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    do_constant_folding=True,
)
mb = os.path.getsize(OUT_ONNX) / 1e6
print('saved %s  (%.1f MB)' % (OUT_ONNX, mb))
assert mb > 10, 'ONNX < 10 MB — weight 누락(external data) 의심!'

# (선택) onnxruntime 로 PyTorch 출력과 일치 검증
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(OUT_ONNX, providers=['CPUExecutionProvider'])
    with torch.no_grad():
        ref = model(dummy).numpy()
    got = sess.run(['logits'], {'input': dummy.numpy()})[0]
    print('onnx vs torch max|Δ|: %.3e' % np.abs(ref - got).max())
except Exception as e:
    print('onnxruntime 검증 skip:', e)

In [ ]:
# 다운로드 (또는 Drive 로 복사)
try:
    from google.colab import files
    files.download(OUT_ONNX)
except Exception as e:
    print('수동:', e)
# import shutil; shutil.copy(OUT_ONNX, '/content/drive/MyDrive/terra/')   # Drive 로 복사하려면 주석 해제

## 7. 다음 (Jetson 배포 — Phase 4)
```bash
scp mobilenet_v3_family_v1.onnx snup2@snup2-desktop:~/terra/
ssh snup2@snup2-desktop "cd ~/terra && trtexec --onnx=mobilenet_v3_family_v1.onnx --fp16 --saveEngine=mnv3_family_v1_fp16.plan --workspace=512"
python3 web_server.py --stm32 --plan mnv3_family_v1_fp16.plan   # unknown threshold(max-prob<0.7) 는 web_server TODO
```
`web/people.json` 라벨 0~4 매핑 확인.

**v2/v3 비교**: `BACKBONE` 만 바꿔 두 번 학습 → 5번 셀 val accuracy + confusion 비교 후 좋은 쪽 배포.